# Clifford data regression

Clifford data regression (CDR) runs a set of **training circuits** with the same structure as the target circuit but made of Clifford gates, so their noiseless values are computed exactly and cheaply. A line fitted from the measured to the exact values of the training circuits is then applied to the measured value of the target.

`ErrorReductionWorkflow` runs in four steps, `plan`, `submit`, `collect` and `estimate`, and each step stores its result on the workflow. The workflow can be saved after any step and reloaded in a later session.

## 0. Setup

In [ ]:
from iqm.error_reduction_tools.cdr import CDR, CDREstimation, plot_cdr_fit, plot_cdr_summary
from iqm.error_reduction_tools.observables import ObservableSet
from iqm.error_reduction_tools.readout_characterization.readout_error_characterization import RECConfiguration
from iqm.error_reduction_tools.twirling.twirling_api import TwirlingConfiguration
from iqm.error_reduction_tools.utils.qiskit_utils import canonicalize_circuit
from iqm.error_reduction_tools.workflow import ErrorReductionWorkflow
from iqm.qiskit_iqm import IQMProvider
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from iqm.iqm_client.util import print_env_vars

See the IQM Resonance documentation for how to connect. The workflow submits the circuits through the backend's client.

In [ ]:
provider = IQMProvider("https://resonance.iqm.tech", quantum_computer="garnet")
backend = provider.get_backend()

## 1. Circuits and observable

Two 6-qubit GHZ circuits with a small `RX` rotation on every qubit, differing in the angle, so neither is Clifford.

The observable is the ring of nearest-neighbour `ZZ` terms, `H = Z0 Z1 + Z1 Z2 + ... + Z5 Z0`. Pauli labels are little-endian: the rightmost character is classical bit 0.

In [ ]:
NUM_QUBITS = 6
OBSERVABLE = [(1.0, "ZZIIII"), (1.0, "IZZIII"), (1.0, "IIZZII"), (1.0, "IIIZZI"), (1.0, "IIIIZZ"), (1.0, "ZIIIIZ")]

def ghz_with_rotation(num_qubits, angle):
    """GHZ chain plus a small RX rotation per qubit, so the circuit is not Clifford."""
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for qubit in range(num_qubits - 1):
        qc.cx(qubit, qubit + 1)
    for qubit in range(num_qubits):
        qc.rx(angle, qubit)
    return qc


def exact_distribution(qc):
    """Return the noiseless output distribution of an unmeasured circuit. Feasible only because it is small."""
    probabilities = Statevector.from_instruction(qc).probabilities()
    return {format(i, f"0{qc.num_qubits}b"): float(p) for i, p in enumerate(probabilities) if p > 1e-15}


angles = [0.35, 0.70]
bare = [ghz_with_rotation(NUM_QUBITS, angle) for angle in angles]

observables = ObservableSet.build([OBSERVABLE])
ideals = [float(observables.combine(observables.term_values_from_counts(exact_distribution(qc)))[0]) for qc in bare]

# The workflow takes iqm.pulse circuits. Transpile to the QPU, then convert, naming qubits after the backend.
targets = []
for angle, qc in zip(angles, bare):
    measured = qc.copy()
    measured.measure_all()
    transpiled = transpile(measured, backend=backend, optimization_level=3)
    pulse_circuit, _ = canonicalize_circuit(transpiled, name=f"ghz_{angle}", backend=backend)
    targets.append(pulse_circuit)

for angle, ideal in zip(angles, ideals):
    print(f"angle {angle:.2f}: ideal energy = {ideal:+.4f}")

## 2. The workflow

`CDR` decides which circuits run on the QPU:

* the number of training circuits, and how they are chosen (`strategy`);
* the shots of the target circuit and of each training circuit;
* the twirling instances of the target and of each training circuit.

A budget left unset in `CDR` comes from the workflow (`shots`) or the twirling configuration (`num_twirling_instances`). A budget set in `CDR` wins, with a warning if you also gave a different value to the workflow or the twirling configuration.

`CDREstimation`, given later to `estimate`, is free to change: whether to fit each Pauli term separately (`combine="per_term"`) or the whole observable (`combine="group"`).

In [ ]:
workflow = ErrorReductionWorkflow(
    targets,
    [OBSERVABLE],
    CDR(
        num_training_circuits=10,
        strategy="pool_fps",
        target_shots=10_000,
        training_shots=100,
        num_target_instances=4,
        num_training_instances=10,
        seed=5,
    ),
    client=backend.client,
    twirling_configuration=TwirlingConfiguration(readout_twirl_strategy="LOCAL", num_twirling_instances=4, seed=42),
    rec_configuration=RECConfiguration(),
)

## 3. Plan, submit, collect

`plan` is free and shows the cost before anything runs. The target and the training circuits use different shots per circuit, so they become separate jobs.

In [ ]:
plan = workflow.plan()
print(plan.summary())

`submit` sends every job to the QPU and returns once they are queued.

In [ ]:
submission = workflow.submit()
print("job ids:", [job.job_id for job in submission.jobs])

`collect` waits for the jobs and keeps the counts. Save the workflow to keep them.

In [ ]:
counts = workflow.collect()
workflow.save("cdr_workflow.json")
print("Collected", len(counts), "count distributions.")

## 4. Estimate

Estimation makes no use of the quantum computer, so try different settings freely. A later session starts by loading the saved workflow.

In [ ]:
workflow = ErrorReductionWorkflow.load("cdr_workflow.json")

result = workflow.estimate(CDREstimation(combine="per_term"))
print("estimate:   ", result.values[:, 0])
print("unmitigated:", result.unmitigated[:, 0])
print("ideal:      ", ideals)

Plotting the fit shows how well the training circuits constrain the line, and whether the target lies inside their range.

In [ ]:
fig = plot_cdr_fit(result)
plt.show()

In [ ]:
result = workflow.estimate(CDREstimation(combine="group"))
fig = plot_cdr_summary(result, ideals=ideals)
plt.show()